# 🚗 Honda HR-V — Buscador de Oportunidades
**Preço:** até R$ 80.000 &nbsp;|&nbsp; **Cor:** qualquer (exceto branco) &nbsp;|&nbsp; **Local:** SP + cidades até ~1h20

Sites consultados: Mercado Livre · WebMotors · iCarros


In [ ]:
# Instala dependências (rode só na primeira vez)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "requests", "pandas", "beautifulsoup4", "lxml", "-q"])
print("✓ Dependências prontas")


In [ ]:
import requests, re, time, unicodedata
from datetime import datetime
import pandas as pd

# ── Configurações — edite aqui ───────────────────────────────────────────────
MAX_PRICE    = 80_000        # R$ teto de preço
MAX_KM_IDEAL = 100_000       # km "bom" para pontuação
ANO_BASE     = 2015          # ano mínimo considerado

CORES_EXCLUIDAS = {"branco", "branca", "white", "off white", "off-white", "pérola", "perola"}

# SP capital + cidades até ~1h20 de carro
CIDADES_ACEITAS = {
    "sao paulo",
    # Grande SP
    "guarulhos","osasco","sao bernardo do campo","santo andre","diadema",
    "maua","ribeirao pires","rio grande da serra","sao caetano do sul",
    "mogi das cruzes","suzano","poa","ferraz de vasconcelos",
    "itaquaquecetuba","aruja","carapicuiba","itapevi","jandira",
    "barueri","santana de parnaiba","pirapora do bom jesus","cotia",
    "embu das artes","embu","taboao da serra","itapecerica da serra",
    "sao lourenco da serra","mairipora","francisco morato",
    "franco da rocha","caieiras","cajamar",
    # Campinas e entorno
    "campinas","valinhos","vinhedo","itatiba","jundiai","louveira",
    "itupeva","indaiatuba","salto","itu","americana","sumare",
    "nova odessa","hortolandia","santa barbara do oeste","paulinia",
    "cosmopolis","artur nogueira","engenheiro coelho","holambra","monte mor",
    # Bragança / Atibaia
    "atibaia","braganca paulista","campo limpo paulista",
    "varzea paulista","jarinu","nazare paulista","piracaia",
    # Sorocaba
    "sorocaba","votorantim","aracoiaba da serra","porto feliz","boituva","tatui",
    # Leste
    "jacaeri","sao jose dos campos",
    # Outros
    "ibiuna","sao roque","mairinque","aluminio","aracariguama",
}

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"),
    "Accept-Language": "pt-BR,pt;q=0.9",
}

GREEN_FLAGS = [
    "único dono","unico dono","1 dono","revisão em dia","revisoes em dia",
    "ipva pago","garantia","sem multa","sem débito","impecável","impecavel",
    "original","placa i","vistoriado",
]
RED_FLAGS = [
    "batido","funilaria","amassado","reformado","sem laudo","sinistro",
    "recuperado","leilão","leilao","financiado","vendo partes","para peças",
]

print("✓ Configurações carregadas")


In [ ]:
def normalizar(texto):
    if not texto: return ""
    s = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def cor_permitida(cor, titulo=""):
    for excluida in CORES_EXCLUIDAS:
        if cor and normalizar(cor) == excluida: return False
        if re.search(rf"\b{re.escape(excluida)}\b", normalizar(cor or "")): return False
    return True

def cidade_permitida(cidade, estado):
    if not cidade: return True
    if estado and normalizar(estado) not in {"sp", "sao paulo", "são paulo"}: return False
    return normalizar(cidade) in CIDADES_ACEITAS

def parse_km(value):
    if value is None: return None
    s = str(value).lower().replace(".","").replace(",","").strip()
    nums = re.findall(r"\d+", s)
    if not nums: return None
    val = int(nums[0])
    if val < 500 and "km" not in s: val *= 1_000
    return val

def fmt_price(v):
    if v is None: return "N/I"
    return f"R$ {v:,.0f}".replace(",","X").replace(".",",").replace("X",".")

def fmt_km(v):
    km = parse_km(v) if not isinstance(v, int) else v
    if km is None: return "N/I"
    return f"{km:,} km".replace(",",".")

def check_flags(text):
    t = (text or "").lower()
    return [f for f in GREEN_FLAGS if f in t], [f for f in RED_FLAGS if f in t]

def score_listing(row):
    score = 0.0
    price = row.get("preco") or MAX_PRICE
    score += max(0, (MAX_PRICE - price) / MAX_PRICE * 35)
    km = parse_km(row.get("km"))
    if km is not None:
        score += max(0, (MAX_KM_IDEAL - km) / MAX_KM_IDEAL * 35)
    try:
        ano = int(str(row.get("ano") or "")[:4])
        score += max(0, min(20, (ano - ANO_BASE) * 3))
    except ValueError:
        pass
    desc = str(row.get("titulo","")) + " " + str(row.get("descricao",""))
    green, red = check_flags(desc)
    score += min(10, len(green) * 3)
    score -= len(red) * 5
    return round(max(0, min(100, score)), 1)

print("✓ Funções auxiliares prontas")


In [ ]:
def buscar_mercadolivre():
    print("🔍 Mercado Livre...")
    resultados, offset, limit = [], 0, 50
    while offset < 250:
        params = {"q":"Honda HR-V","category":"MLB1744",
                  "price_to":MAX_PRICE,"offset":offset,"limit":limit}
        try:
            r = requests.get("https://api.mercadolibre.com/sites/MLB/search",
                             params=params, headers=HEADERS, timeout=15)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f"  ⚠ Erro: {e}"); break

        items = data.get("results", [])
        total = data.get("paging", {}).get("total", 0)
        if not items: break

        for item in items:
            if not re.search(r"hr.?v", item.get("title",""), re.IGNORECASE): continue
            attrs = {a["id"]: a.get("value_name") for a in item.get("attributes",[])}
            resultados.append({
                "fonte":"Mercado Livre",
                "titulo": item.get("title",""),
                "preco": item.get("price"),
                "km":    attrs.get("VEHICLE_MILEAGE") or attrs.get("KILOMETERS"),
                "ano":   attrs.get("VEHICLE_YEAR"),
                "versao":attrs.get("TRIM") or attrs.get("MODEL"),
                "cambio":attrs.get("VEHICLE_TRANSMISSION"),
                "cor":   attrs.get("COLOR"),
                "cidade":item.get("address",{}).get("city_name",""),
                "estado":item.get("address",{}).get("state_name",""),
                "descricao":"",
                "link":  item.get("permalink",""),
            })
        offset += limit
        if offset >= total: break
        time.sleep(0.4)
    print(f"  → {len(resultados)} anúncios encontrados")
    return resultados

resultados_ml = buscar_mercadolivre()


In [ ]:
def buscar_webmotors():
    print("🔍 WebMotors...")
    resultados = []
    base = ("https://www.webmotors.com.br/carros/estoque/"
            "?TipoVeiculo=carros&Marca=HONDA&Modelo=HR-V&PrecoAte=80000")
    for page in range(1, 6):
        params = {"url":f"{base}&Pag={page}","actualPage":page,
                  "displayPerPage":24,"order":1,"showMenu":"true",
                  "listFilters":"true","startZero":"false"}
        try:
            r = requests.get("https://www.webmotors.com.br/api/search/car",
                             params=params, headers=HEADERS, timeout=15)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f"  ⚠ Erro p.{page}: {e}"); break

        anuncios = data.get("SearchResults", [])
        if not anuncios: break
        for a in anuncios:
            spec   = a.get("Specification", {})
            prices = a.get("Prices", {})
            preco  = prices.get("Price") or prices.get("PriceValue")
            if preco and preco > MAX_PRICE: continue
            titulo = (f"{spec.get('Make','')} {spec.get('Model','')} "
                      f"{spec.get('Version','')} {spec.get('ModelYear','')}").strip()
            resultados.append({
                "fonte":"WebMotors", "titulo":titulo, "preco":preco,
                "km":   spec.get("OdometerLastValue"),
                "ano":  spec.get("ModelYear") or spec.get("YearFabrication"),
                "versao":spec.get("Version"), "cambio":spec.get("GearShift"),
                "cor":  spec.get("Color"),
                "cidade":a.get("Seller",{}).get("City",""),
                "estado":a.get("Seller",{}).get("State",""),
                "descricao":"",
                "link": f"https://www.webmotors.com.br/carros/anuncio/{a.get('UniqueId','')}",
            })
        if len(anuncios) < 24: break
        time.sleep(0.5)
    print(f"  → {len(resultados)} anúncios encontrados")
    return resultados

resultados_wm = buscar_webmotors()


In [ ]:
def buscar_icarros():
    print("🔍 iCarros...")
    from bs4 import BeautifulSoup
    resultados = []
    for page in range(1, 4):
        url = (f"https://www.icarros.com.br/ache/listaanuncios.jsp"
               f"?pag={page}&ord=2&modelo=5&marca=21&priceMax={MAX_PRICE}")
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "lxml")
        except Exception as e:
            print(f"  ⚠ Erro p.{page}: {e}"); break

        cards = soup.select("li.ads__list__item") or soup.select(".anuncio")
        if not cards: break
        for card in cards:
            try:
                titulo = (card.select_one("[class*='title']") or card.select_one("h2")).get_text(strip=True)
                preco_el = card.select_one("[class*='price']") or card.select_one(".preco")
                nums = re.findall(r"\d+", (preco_el.get_text(strip=True) if preco_el else "").replace(".",""))
                preco = int(nums[0]) if nums else None
                km_el = card.select_one("[class*='km']") or card.select_one(".km")
                km_txt = km_el.get_text(strip=True) if km_el else ""
                link_el = card.select_one("a[href]")
                link = link_el["href"] if link_el else ""
                if link and not link.startswith("http"): link = "https://www.icarros.com.br" + link
                resultados.append({"fonte":"iCarros","titulo":titulo,"preco":preco,
                                   "km":km_txt,"ano":None,"versao":None,"cambio":None,
                                   "cor":None,"cidade":"","estado":"","descricao":"","link":link})
            except: continue
        time.sleep(0.5)
    print(f"  → {len(resultados)} anúncios encontrados")
    return resultados

resultados_ic = buscar_icarros()


In [ ]:
todos = resultados_ml + resultados_wm + resultados_ic

# Remove duplicados
vistos, unicos = set(), []
for item in todos:
    chave = item.get("link","").split("?")[0]
    if chave and chave not in vistos:
        vistos.add(chave); unicos.append(item)

total_bruto = len(unicos)

# Filtro cor
sem_branco = [x for x in unicos if cor_permitida(x.get("cor"), x.get("titulo",""))]

# Filtro cidade
filtrados = [x for x in sem_branco if cidade_permitida(x.get("cidade"), x.get("estado"))]

print(f"Total coletado : {total_bruto}")
print(f"Removidos (cor branca)  : {total_bruto - len(sem_branco)}")
print(f"Removidos (fora da área): {len(sem_branco) - len(filtrados)}")
print(f"Válidos para análise    : {len(filtrados)}")


In [ ]:
df = pd.DataFrame(filtrados)
if df.empty:
    print("⚠ Nenhum anúncio encontrado após filtros.")
else:
    df["score"]  = df.apply(score_listing, axis=1)
    df["km_num"] = df["km"].apply(parse_km)
    df["preco_fmt"] = df["preco"].apply(fmt_price)
    df["km_fmt"]    = df["km"].apply(fmt_km)

    ranking = df.sort_values("score", ascending=False).reset_index(drop=True)
    ranking.index += 1

    print(f"\n{'='*65}")
    print(f"  🏆 TOP {min(25, len(ranking))} MELHORES OPORTUNIDADES")
    print(f"{'='*65}\n")

    for i, row in ranking.head(25).iterrows():
        barra = "█" * int(row.score/10) + "░" * (10 - int(row.score/10))
        green, red = check_flags(str(row.get("titulo","")) + " " + str(row.get("descricao","")))
        print(f"#{i:02d} [{barra}] Score {row.score:5.1f}/100  [{row.fonte}]")
        print(f"    {row.titulo}")
        print(f"    Preço : {row.preco_fmt}  |  KM : {row.km_fmt}  |  Ano : {row.get('ano','N/I')}")
        if row.get("versao"): print(f"    Versão: {row.versao}  |  Câmbio: {row.get('cambio','N/I')}  |  Cor: {row.get('cor','N/I')}")
        local = f"{row.get('cidade','')} - {row.get('estado','')}".strip(" -")
        if local: print(f"    Local : {local}")
        if green: print(f"    ✅ {' | '.join(green)}")
        if red:   print(f"    ⚠️  {' | '.join(red)}")
        print(f"    🔗 {row.link}")
        print()


In [ ]:
if not df.empty:
    colunas = ["titulo","preco_fmt","km_fmt","ano","versao","cor","cidade","score","link"]
    colunas_presentes = [c for c in colunas if c in ranking.columns]
    display(ranking[colunas_presentes].head(25).style
            .background_gradient(subset=["score"], cmap="RdYlGn")
            .set_caption("Honda HR-V até R$80k — melhores negócios"))


In [ ]:
if not df.empty:
    precos = ranking["preco"].dropna()
    kms    = ranking["km_num"].dropna()
    print(f"{'='*45}")
    print("  RESUMO ESTATÍSTICO")
    print(f"{'='*45}")
    if len(precos):
        print(f"  Preço médio  : {fmt_price(int(precos.mean()))}")
        print(f"  Mais barato  : {fmt_price(int(precos.min()))}")
        print(f"  Mais caro    : {fmt_price(int(precos.max()))}")
    if len(kms):
        print(f"  KM médio     : {fmt_km(int(kms.mean()))}")
        print(f"  Menor KM     : {fmt_km(int(kms.min()))}")
    print(f"  Anúncios OK  : {len(ranking)}")
    print()
    print("Distribuição por fonte:")
    print(ranking["fonte"].value_counts().to_string())
